In [42]:
import requests
import json
from dotenv import load_dotenv
import os
import re
from datetime import datetime

load_dotenv()
custom_request_header = os.getenv("CUSTOM_REQUEST_HEADER")

In [43]:
USER_API_URL = 'http://localhost:3000/api/users'
HEADERS = { "x-customrequired-header": custom_request_header }

In [44]:
# Get a List of all users
r = requests.get(USER_API_URL, headers=HEADERS)
users = json.loads(r.content)

In [45]:
user_dict = {}
for user in users:
    user_dict[user['_id']] = user

In [46]:
def identify_problem_users(users):
    users_with_capital_emails = [user for user in users if re.compile('[A-Z]').search(user['email'])]

    potential_duplicate_emails = set([user['email'].lower() for user in users_with_capital_emails])

    problem_users = {}
    for user in users:
        current_email = user['email'].lower()
        if current_email in potential_duplicate_emails:
            if problem_users.get(current_email, None) is not None:
                problem_users[current_email].append(user['_id'])
            else:
                problem_users[current_email] = [user['_id']]

    non_duped_capital_emails_with_ids = set([(email, problem_users[email][0]) for email in problem_users.keys() if len(problem_users[email]) == 1])

    for email, user_id in non_duped_capital_emails_with_ids:
        problem_users.pop(email)

    return problem_users, non_duped_capital_emails_with_ids

In [47]:
duped_emails, non_duped_capital_emails_with_ids = identify_problem_users(users)

In [48]:
duped_emails

{'atkendis@gmail.com': ['5e27b1e54530cd0017eee431',
  '5e30ee300b9d2300177d3b3a'],
 'eli.j.selkin@gmail.com': ['5e7965101e29ad00179399bb',
  '5e7024f88e1bee00178aa1e0'],
 'jasc68.jyang@gmail.com': ['5e38d1568d52770017ae8a86',
  '5e66e6fcbe3e0b001761a814'],
 'trillium@hatsfabulous.com': ['5e965e554e2fc70017aa3970',
  '633b9a74d98663001f8b5c46'],
 'dannyprikaz@gmail.com': ['678f122e4c6f61002a1e5e68',
  '6871afe1e6bf590aede8f8da',
  '6871b00ee6bf590aede8f8db']}

In [49]:
non_duped_capital_emails_with_ids

{('acanderson358@gmail.com', '5e4c995eb73a2a001732f47e'),
 ('akib.rhasast@gmail.com', '60d6a4e82e675e4a90e9ca92'),
 ('christopher.a.schmitz@gmail.com', '5e30ee120b9d2300177d3b38'),
 ('dannydaekim@gmail.com', '5f28c25206f21000177e690a'),
 ('dedre@sharklasers.com', '6101d94c4815993498437083'),
 ('dpease@dataminr.com', '5e1d2283316d2f00172ef05a'),
 ('jbarib@gmail.com', '5e27abf74530cd0017eee417'),
 ('juliamzfong@gmail.com', '5e435911ef67e100175c1ecb'),
 ('kphowley@gmail.com', '5e4c9b59b73a2a001732f487'),
 ('rmcollins95@gmail.com', '5e38d10a8d52770017ae8a81'),
 ('scott@scottlarsen.com', '5e435ce7ef67e100175c1ee3'),
 ('tywe@sharklasers.com', '6101db234815993498437084')}

In [50]:
def update_user_email(email, user_id):
    r = requests.patch(USER_API_URL + '/' + user_id, json={'email': email}, headers=HEADERS)
    print(r.content)
    
def fix_non_duped_capital_emails(emails_with_ids):
    for email, user_id in emails_with_ids:
        print(email, user_id)
        update_user_email(user_id, email)

In [ ]:
users[0]

In [51]:
def determine_canonical_id_for_duped_email(duped_email_ids, user_dict):
    canonical_id = ''
    oldest_creation_date = datetime.now()
    for user_id in duped_email_ids:
        current_creation_date = datetime.strptime(user_dict[user_id]['createdDate'], '%Y-%m-%dT%H:%M:%S.%fZ')
        if current_creation_date < oldest_creation_date:
            oldest_creation_date = current_creation_date
            canonical_id = user_id

    return canonical_id

In [52]:
determine_canonical_id_for_duped_email(duped_emails['dannyprikaz@gmail.com'], user_dict)

'678f122e4c6f61002a1e5e68'

In [34]:
def merge_users(canonical_user_id, user_id_2, user_dict):
    canonical_user = user_dict[canonical_user_id]
    duplicate_user = user_dict[user_id_2]

    # For any list fields, combine them
    list_fields = ['skillsToMatch', 'projects', 'managedProjects']
    for field in list_fields:
        canonical_user[field] += duplicate_user[field]

    # For boolean fields, set to true if either is true
    bool_fields = ['textingOk', 'isActive', 'newMember']
    for field in bool_fields:
        canonical_user[field] = canonical_user[field] or duplicate_user[field]

    # For fields about roles, take the most recent information
    take_the_newer_fields = ['currentRole', 'desiredRole']
    for field in take_the_newer_fields:
        if len(duplicate_user.get(field, '')) > 0:
            canonical_user[field] = duplicat_user[field]

    # Take the highest access level
    access_level = ['user', 'admin', 'superadmin']
    highest_access_level = max(access_level.index(canonical_user['accessLevel']), access_level.index(duplicate_user['accessLevel']))
    canonical_user['accessLevel'] = access_level[highest_access_level]

    # Make user that email is all lower case
    canonical_user['email'] = canonical_user['email'].lower()
    
    
    print(canonical_user)

In [23]:
merge_users(duped_emails['dannyprikaz@gmail.com'][0], duped_emails['dannyprikaz@gmail.com'][1], user_dict)

{'name': {'firstName': 'Danny', 'lastName': 'Prikazsky'}, 'accessLevel': 'admin', 'skillsToMatch': [], 'projects': [], 'textingOk': False, 'managedProjects': [], 'isActive': True, '_id': '678f122e4c6f61002a1e5e68', 'email': 'dannyprikaz@gmail.com', 'currentRole': 'Full Stack Developer', 'desiredRole': 'Full Stack Developer', 'newMember': True, 'firstAttended': 'JAN 2025', 'createdDate': '2025-01-21T03:19:10.548Z', '__v': 0}


In [36]:
user_dict[duped_emails['dannyprikaz@gmail.com'][2]]

{'name': {'firstName': 'Danny', 'lastName': 'Prikazsky'},
 'accessLevel': 'admin',
 'skillsToMatch': [],
 'projects': [],
 'textingOk': False,
 'managedProjects': [],
 'isActive': True,
 '_id': '6871b00ee6bf590aede8f8db',
 'email': 'dannypRikaz@gmail.com',
 'currentRole': 'Full Stack Developer',
 'desiredRole': 'Full Stack Developer',
 'newMember': True,
 'firstAttended': 'JAN 2025',
 'createdDate': '2025-01-21T03:19:10.548Z',
 '__v': 0}

In [39]:
r = requests.get('http://localhost:3000/api/projectteammembers', headers=HEADERS)

# We need to find all of the other documents in the database that reference the duplicated users.

### Database objects that reference User Ids:
- `checkIn`: checkIns have a userId and an eventId, but the API does not expose endpoints for deleting or updating checkIns. We could keep the IDs of every checkIn with a duplicated user and run them through a script that directly accesses the mongooes object, or we could attempt to access Mongo directly in this script
- `event`: events have a field for owner with an ownerId, which is supposed to be "id of user who created event." None of the events in the database have the owner field filled in.
- `project`: projects have a field called `managedByUsers`. For most documents in the database, this is an empty array. For the ones where it is not an empty array, it does not contain any valid userIds.
- `projectTeamMember`: This seems like it should have plenty of data containing userIds, but there are no projectTeamMember documents in the database
- `recurringEvent`: Similar to event, this has a field for owner with ownerId. It is set to default to '123456', and there are no documents in the database that don't have that value.
- 